*0.4 Deep learning basics*

# Transformer: encoder

**The situation.** The recurrent sentiment models (items 5–7) read one token at a time and reached ~78%. A transformer encoder reads all tokens at once, lets each token attend to every other, and trains in parallel. Same data, same budget — does it do better?

**The encoder.** A stack of identical layers. Each layer: multi-head self-attention (every token looks at every token) → add & normalise → a small feed-forward network on each token → add & normalise. Positions are added at the input. For classification, pool the outputs (mean, or a special first token as BERT does) and add a linear head. BERT is this, 12 layers deep.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# SST-2: 67k movie-review sentences labelled positive/negative — the standard small sentiment set.
sst2 = load_dataset("stanfordnlp/sst2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def encode(rows, max_length=32):
    encoded = tokenizer(
        rows["sentence"], truncation=True, max_length=max_length, padding="max_length"
    )
    return {"ids": encoded["input_ids"], "label": rows["label"]}


train_rows = sst2["train"].shuffle(seed=0).select(range(8000)).map(encode, batched=True)
val_rows = sst2["validation"].map(encode, batched=True)
train_ids = torch.tensor(train_rows["ids"])
train_labels = torch.tensor(train_rows["label"])
val_ids = torch.tensor(val_rows["ids"])
val_labels = torch.tensor(val_rows["label"])
print(
    "train:",
    tuple(train_ids.shape),
    "| validation:",
    tuple(val_ids.shape),
    "| vocabulary:",
    tokenizer.vocab_size,
)

train: (8000, 32) | validation: (872, 32) | vocabulary: 30522


In [3]:
import time

import torch.nn.functional as F
from torch import nn


class EncoderClassifier(nn.Module):
    def __init__(self, vocabulary_size, d_model=64, heads=4, layers=2, max_length=32):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, d_model, padding_idx=0)
        self.positions = nn.Embedding(max_length, d_model)  # learned positions, as BERT does
        layer = nn.TransformerEncoderLayer(
            d_model, heads, dim_feedforward=128, dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers, enable_nested_tensor=False)
        self.output = nn.Linear(d_model, 2)

    def forward(self, token_ids):
        padding = token_ids == 0  # attention must ignore padding
        x = self.embedding(token_ids) + self.positions(torch.arange(token_ids.shape[1]))
        x = self.encoder(x, src_key_padding_mask=padding)
        real = (~padding).unsqueeze(-1).float()
        pooled = (x * real).sum(dim=1) / real.sum(dim=1)  # mean over real tokens
        return self.output(pooled)


def train(model, epochs=3, lr=2e-3, batch_size=64):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(train_ids))
        total_loss = 0.0
        for start in range(0, len(order), batch_size):
            batch = order[start : start + batch_size]
            loss = F.cross_entropy(model(train_ids[batch]), train_labels[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)
        print(
            
                f"epoch {epoch}  train loss {total_loss / len(order):.3f}  val accuracy "
                f"{accuracy(model):.1%}"
            
        )
    return accuracy(model)


def accuracy(model):
    model.eval()
    with torch.no_grad():
        predictions = model(val_ids).argmax(dim=1)
    return (predictions == val_labels).float().mean().item()


torch.manual_seed(0)
started = time.perf_counter()
encoder_accuracy = train(EncoderClassifier(tokenizer.vocab_size), epochs=6, lr=5e-4)
print(f"encoder: {encoder_accuracy:.1%} in {time.perf_counter() - started:.0f} s")
assert encoder_accuracy > 0.6

epoch 1  train loss 0.684  val accuracy 54.2%


epoch 2  train loss 0.658  val accuracy 61.8%


epoch 3  train loss 0.620  val accuracy 61.4%


epoch 4  train loss 0.567  val accuracy 65.7%


epoch 5  train loss 0.506  val accuracy 66.5%


epoch 6  train loss 0.442  val accuracy 66.7%
encoder: 66.7% in 15 s


**Reading the output.** A two-layer encoder trained from scratch on 8,000 sentences lands *below* the LSTM. That is the honest result: transformers are data-hungry, and small data does not show their advantage. What the code shows is the *scaling* property — every position is processed in parallel — so the same architecture on a GPU with millions of sentences becomes BERT (item 16 uses a pretrained one and beats everything here).

```
tokens + positions
   │
   ▼  ┌────────────────────────────────────────────┐
   │  │ multi-head self-attention ─▶ add & norm    │  × N layers
   │  │ feed-forward per token    ─▶ add & norm    │
   ▼  └────────────────────────────────────────────┘
mean-pool ─▶ linear ─▶ class scores
```

**The rule to remember.** Encoder = self-attention + feed-forward, stacked, all positions at once. It produces a contextual vector per token; pool them for classification, keep them for tagging or retrieval.

| Use it when | Don't when | Instead use |
|---|---|---|
| classification, tagging, embeddings — reading tasks | generating text | decoder (next item) |

**Watch out**
- Pass the padding mask, or attention reads the zeros and accuracy drops quietly.
- From-scratch encoders need far more data than this; in practice start from a pretrained one (item 16).
- `nn.TransformerEncoderLayer` defaults to post-norm; modern models use `norm_first=True` — more stable for deep stacks.